# Basic Trace Evaluation

This notebook shows how to fetch production traces from Langfuse and evaluate them with fasteval's LLM-as-judge metrics.

**What you'll learn:**

1. **First Evaluation** — Minimal `@langfuse_traces` + `@fe.correctness` example
2. **Function Parameters** — What `trace_id`, `input`, `output`, `context`, `metadata` contain
3. **Filtering Traces** — By tags, time range, user ID, session ID
4. **Stacking Metrics** — Combine correctness + faithfulness + toxicity in one test
5. **Handling Missing Data** — Gracefully skip traces with incomplete data
6. **RAG Evaluation** — Evaluate retrieval-augmented generation traces
7. **Score Reporting** — How scores appear in Langfuse and how to customize them

Every cell is **runnable**. Make sure you've completed the [Overview notebook](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/overview.ipynb) first for installation and API key setup.

---

## Setup

If you haven't already, install the packages and set your API keys.

In [ ]:
!pip install -q fasteval-core fasteval-langfuse

In [ ]:
import os

# Load from Colab Secrets or set manually
try:
    from google.colab import userdata
    os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY")
    os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY")
    os.environ["LANGFUSE_HOST"] = userdata.get("LANGFUSE_HOST")
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Keys loaded from Colab Secrets.")
except (ImportError, Exception):
    # Uncomment and set manually if not using Colab Secrets
    # os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
    # os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
    # os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"
    # os.environ["OPENAI_API_KEY"] = "sk-..."
    print("Using environment variables.")

In [ ]:
import fasteval as fe
from fasteval_langfuse import langfuse_traces, configure_langfuse, LangfuseConfig

configure_langfuse(LangfuseConfig(
    auto_push_scores=True,
    score_name_prefix="fasteval_",
))

print("fasteval + langfuse ready.")

---

## 1. Your First Trace Evaluation

The simplest evaluation: fetch a few recent traces and check if the AI's response is correct given the user's input.

How it works:
1. `@langfuse_traces` fetches traces matching your filters
2. Your test function is called **once per trace** with the extracted data
3. `@fe.correctness` uses an LLM judge to score the output
4. Scores are automatically pushed back to Langfuse

> **Colab Note:** Start with `limit=3` to avoid burning through your OpenAI budget while experimenting. Scale up once you're happy with the setup.

In [ ]:
@fe.correctness(threshold=0.7)
@langfuse_traces(
    time_range="last_24h",
    limit=3,
)
def test_basic_correctness(trace_id, input, output, context, metadata):
    """Check if each trace's output is correct given the input."""
    print(f"Trace:  {trace_id[:16]}...")
    print(f"  Input:  {str(input)[:100]}")
    print(f"  Output: {str(output)[:100]}")
    print()
    fe.score(output, input=input)


print("Fetching traces...\n")
test_basic_correctness()

---

## 2. Understanding Function Parameters

When `@langfuse_traces` calls your function, it passes these parameters (all extracted automatically from the Langfuse trace):

| Parameter | Type | Description | Extraction Logic |
|-----------|------|-------------|------------------|
| `trace_id` | `str` | Unique trace ID | `trace.id` |
| `input` | `str` | User query/prompt | Auto-extracted from `trace.input` — handles strings and dicts with keys like `query`, `prompt`, `message` |
| `output` | `str` | AI response | Auto-extracted from `trace.output` — handles strings and dicts with keys like `response`, `answer`, `completion` |
| `context` | `str \| None` | Retrieved documents (for RAG) | Extracted from metadata keys: `context`, `retrieved_docs`, `documents`, `retrieval_context` |
| `metadata` | `dict` | Full trace metadata | `trace.metadata` — everything else lives here |

You only need to declare the parameters you care about. For example, if you don't need `context`, just leave it out of the function signature.

Let's inspect what these look like for real traces:

In [ ]:
import json


@langfuse_traces(
    time_range="last_7d",
    limit=2,
)
def inspect_trace_data(trace_id, input, output, context, metadata):
    """Inspect the raw data extracted from each trace."""
    print(f"{'='*60}")
    print(f"Trace ID: {trace_id}")
    print(f"{'='*60}")
    print(f"\nInput ({type(input).__name__}):")
    print(f"  {str(input)[:200]}")
    print(f"\nOutput ({type(output).__name__}):")
    print(f"  {str(output)[:200]}")
    print(f"\nContext: {str(context)[:200] if context else 'None (not a RAG trace)'}")
    print(f"\nMetadata keys: {list(metadata.keys()) if metadata else 'None'}")
    if metadata:
        print(f"  Sample: {json.dumps(dict(list(metadata.items())[:3]), indent=2, default=str)}")
    print()


inspect_trace_data()

---

## 3. Filtering Traces

The `@langfuse_traces` decorator accepts several parameters to control which traces are fetched.

| Parameter | Type | Example | Description |
|-----------|------|---------|-------------|
| `project` | `str` | `"production"` | Filter to a specific Langfuse project |
| `filter_tags` | `list[str]` | `["rag", "v2"]` | Only traces with ALL these tags |
| `time_range` | `str` | `"last_1h"`, `"last_7d"` | Relative time range |
| `time_range` | `str` | `"2026-03-01 to 2026-03-05"` | Absolute time range |
| `user_id` | `str` | `"user-123"` | Filter by user |
| `session_id` | `str` | `"sess-abc"` | Filter by conversation session |
| `limit` | `int` | `100` | Max traces to fetch (before sampling) |

> **Colab Note:** Filtering reduces the number of traces fetched, which directly reduces cost. Always be as specific as possible.

In [ ]:
# Example: Filter by tags and time range
@fe.correctness(threshold=0.7)
@langfuse_traces(
    filter_tags=["customer-support"],
    time_range="last_7d",
    limit=5,
)
def test_support_traces(trace_id, input, output, metadata):
    """Evaluate only customer-support traces from the last week."""
    print(f"  [{trace_id[:12]}] {str(input)[:60]}")
    fe.score(output, input=input)


print("Evaluating customer-support traces...\n")
test_support_traces()

In [ ]:
# Example: Absolute time range for post-incident analysis
@fe.correctness(threshold=0.7)
@langfuse_traces(
    time_range="2026-03-04 14:00 to 2026-03-04 16:00",
    limit=20,
)
def test_incident_window(trace_id, input, output, metadata):
    """Evaluate traces from a specific incident window."""
    print(f"  [{trace_id[:12]}] {str(input)[:60]}")
    fe.score(output, input=input)


print("Evaluating incident window traces...\n")
test_incident_window()

---

## 4. Stacking Multiple Metrics

Stack multiple metric decorators to evaluate different quality dimensions on the same trace. Each decorator generates its own score.

Common combinations:

| Use Case | Metrics |
|----------|--------|
| General quality | `correctness` + `relevance` |
| RAG pipeline | `faithfulness` + `answer_correctness` + `contextual_precision` |
| Safety-critical | `correctness` + `toxicity` + `bias` |
| Customer support | `correctness` + `relevance` + `hallucination` |

> **Colab Note:** Each metric = 1 LLM judge call per trace. Stacking 3 metrics on 100 traces = 300 LLM calls. Plan accordingly!

In [ ]:
# Stack correctness + relevance + hallucination
@fe.correctness(threshold=0.8)
@fe.relevance(threshold=0.8)
@fe.hallucination(threshold=0.9)
@langfuse_traces(
    time_range="last_24h",
    limit=3,
)
def test_comprehensive_quality(trace_id, input, output, context, metadata):
    """Multi-dimensional quality check."""
    print(f"Evaluating trace {trace_id[:12]}... (3 metrics)")
    fe.score(output, context=context, input=input)


print("Running comprehensive evaluation...\n")
test_comprehensive_quality()

When `auto_push_scores=True`, each metric generates a separate score in Langfuse:
- `fasteval_correctness`
- `fasteval_relevance`
- `fasteval_hallucination`
- `fasteval_aggregate` (average of all metrics)

---

## 5. Handling Missing Data

Production traces can be messy. Some might have missing inputs, empty outputs, or no context. Here are patterns for handling this gracefully.

> **Colab Note:** Without these guards, a single malformed trace could cause your entire evaluation to fail. Always add safety checks in production evaluations.

In [ ]:
@fe.correctness(threshold=0.7)
@langfuse_traces(
    time_range="last_7d",
    limit=10,
)
def test_robust_evaluation(trace_id, input, output, context, metadata):
    """Gracefully handle missing or malformed trace data."""

    # Guard 1: Skip traces with missing essential data
    if not input or not output:
        print(f"  SKIP {trace_id[:12]}: missing input or output")
        return

    # Guard 2: Skip very short outputs (likely errors)
    if len(str(output)) < 10:
        print(f"  SKIP {trace_id[:12]}: output too short ({len(str(output))} chars)")
        return

    # Guard 3: Fallback for context if auto-extraction missed it
    if not context and metadata:
        context = (
            metadata.get("custom_docs_key")
            or metadata.get("retrieved_documents")
            or metadata.get("sources")
        )

    print(f"  EVAL {trace_id[:12]}: input={len(str(input))}chars, output={len(str(output))}chars, context={'yes' if context else 'no'}")
    fe.score(output, context=context, input=input)


print("Running robust evaluation...\n")
test_robust_evaluation()

---

## 6. RAG Evaluation

For Retrieval-Augmented Generation (RAG) applications, you can use specialized metrics that evaluate both the retrieval quality and the generation quality.

**RAG-specific metrics:**

| Metric | What it measures |
|--------|------------------|
| `faithfulness` | Is the response grounded in the retrieved context? (no hallucinations) |
| `answer_correctness` | Does the response correctly answer the question? |
| `contextual_precision` | Are the retrieved docs relevant to the question? |
| `contextual_recall` | Do the retrieved docs cover what's needed to answer? |
| `relevance` | Is the response relevant to the user's question? |

These metrics require `context` to be available in the trace. The plugin auto-extracts context from metadata keys like `context`, `retrieved_docs`, `documents`, and `retrieval_context`.

> **Colab Note:** RAG metrics are the most powerful quality signals for RAG applications. `faithfulness` catches hallucinations, while `contextual_precision` catches irrelevant retrieval.

In [ ]:
# Full RAG evaluation stack
@fe.faithfulness(threshold=0.9)
@fe.answer_correctness(threshold=0.8)
@fe.relevance(threshold=0.8)
@langfuse_traces(
    filter_tags=["rag"],
    time_range="last_7d",
    limit=5,
)
def test_rag_pipeline(trace_id, input, output, context, metadata):
    """Full RAG quality evaluation."""
    if not context:
        # Fallback: try to get context from common metadata keys
        context = metadata.get("docs") or metadata.get("retrieval_context")

    if not context:
        print(f"  SKIP {trace_id[:12]}: no context available for RAG metrics")
        return

    print(f"  EVAL {trace_id[:12]}: {len(str(context))} chars of context")

    expected = metadata.get("expected_answer")
    fe.score(output, expected, context=context, input=input)


print("Running RAG evaluation...\n")
test_rag_pipeline()

---

## 7. Score Reporting

When `auto_push_scores=True` (the default), fasteval pushes evaluation results back to Langfuse as **scores** on each trace.

### What gets pushed

For each evaluated trace, these scores are created:

| Score Name | Value | Description |
|------------|-------|-------------|
| `fasteval_correctness` | 0.0–1.0 | Individual metric score |
| `fasteval_faithfulness` | 0.0–1.0 | Individual metric score |
| `fasteval_aggregate` | 0.0–1.0 | Average of all metrics |

Each score also includes the **judge's reasoning** in the comment field.

### Viewing scores in Langfuse

1. Go to **Langfuse > Traces**
2. You'll see new score columns for each metric
3. **Filter** by score: `score.fasteval_correctness < 0.7` to find issues
4. Click a trace to see the full score breakdown and reasoning

### Customizing the prefix

Change the prefix to organize scores from different evaluation pipelines:

In [ ]:
# Example: Custom prefix for different evaluation contexts

# For nightly deep-dive evaluations
configure_langfuse(LangfuseConfig(
    score_name_prefix="nightly_",
    auto_push_scores=True,
))
# Scores become: nightly_correctness, nightly_aggregate
print("Prefix set to 'nightly_'")
print("  Scores will appear as: nightly_correctness, nightly_faithfulness, etc.")

# Reset to default
configure_langfuse(LangfuseConfig(
    score_name_prefix="fasteval_",
    auto_push_scores=True,
))
print("\nReset to default prefix 'fasteval_'")

In [ ]:
# Example: Disable score pushing for local-only evaluation
@fe.correctness(threshold=0.7)
@langfuse_traces(
    time_range="last_24h",
    limit=2,
    auto_push_scores=False,  # Override: don't push to Langfuse
)
def test_local_only(trace_id, input, output, metadata):
    """Evaluate locally without pushing scores to Langfuse."""
    print(f"  Evaluating {trace_id[:12]}... (scores NOT pushed to Langfuse)")
    fe.score(output, input=input)


print("Local-only evaluation (no scores pushed):\n")
test_local_only()

---

## 8. Running with pytest

In production, you'll run these evaluations with `pytest` rather than in a notebook. Here's how the same code looks as a test file:

```python
# tests/test_production_eval.py
import fasteval as fe
from fasteval_langfuse import langfuse_traces
from fasteval_langfuse.sampling import RandomSamplingStrategy


@fe.correctness(threshold=0.8)
@fe.hallucination(threshold=0.9)
@langfuse_traces(
    project="production",
    filter_tags=["customer-support"],
    time_range="last_24h",
    sampling=RandomSamplingStrategy(sample_size=200, seed=42),
)
def test_production_quality(trace_id, input, output, context, metadata):
    expected = metadata.get("expected_answer")
    fe.score(output, expected, context=context, input=input)
```

Run it:

```bash
pytest tests/test_production_eval.py -v
```

Output:
```
Evaluating 200/5,432 traces (3.7% sample, strategy=RandomSamplingStrategy)
test_production_quality PASSED

Scores pushed to Langfuse with prefix "fasteval_"
```

---

## Next Steps

| Notebook | What you'll learn |
|----------|-------------------|
| [Sampling Strategies](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/sampling-strategies.ipynb) | All 5 sampling strategies, cost optimization, GitHub Actions automation |
| [Dataset Regression Testing](https://colab.research.google.com/github/intuit/fasteval/blob/main/docs/notebooks/langfuse-trace-evaluation/dataset-regression-testing.ipynb) | Golden datasets, A/B testing, Langfuse Experiments, CI/CD gates |

For the full API reference, see the [fasteval-langfuse plugin docs](https://fasteval.dev/docs/plugins/langfuse/overview).